# Test of G1 (`01_repository_delivery_metrics`)

This notebook tests `01_repository_delivery_metrics` with a small hand-made Silver table in the schema `github_lakehouse.test_gold_g1`. It runs the real notebook with `%run` and `test_schema = test_gold_g1`, so **production tables are never written**. The last check section proves that.

It has the same sections as the test of G2. Only the hand-made rows and the expected values are different.

**Before you run it:** `08_gold_audit_table` must have run once, and the Silver table `silver.pull_request_events` must exist. The test copies the definitions of both tables. It does not read their rows.

## How to read the output

- Every check section prints a table with one line per check: `PASS` or `FAIL`, the actual value and the expected value.
- A section stops with an error when one of its checks failed.
- **Four cells are EXPECTED to fail.** The text before each of them says so. "Run all" stops at such a cell. Then click the cell **after** it and choose "Run all below". It does no harm if one of these four cells runs more than once.
- **The result of the whole test is the cell "Result of the whole test" near the end.** It prints `TEST PASSED` only when all 13 sections ran and every check passed. If a section failed or was left out, it stops with `TEST FAILED` and names the section.
- The last cell drops the test schema.

## What is tested

| Test of the contract | Where |
|---|---|
| No input | T3, T4: an empty Silver table. T14, T15: a source date without rows. Both give `NO_INPUT` and leave Gold unchanged |
| Schema | T7: the Gold table has exactly the columns and types of the contract |
| Exact values | T7: hand-made rows give the hand-calculated Gold rows |
| Missing values | T7: a close event without a merged flag or without usable times, an unknown action, a pull request without a key, a snapshot without a size, an event without a name, an event without a date |
| Zero base | T7: a day without a close event has NULL for the share and the durations, and a day without a known size has NULL for the size averages, not 0 |
| Rerun | T8, T9: a second run changes nothing |
| Late event | T10, T11: new Silver rows change the right Gold rows and the right snapshot, and the other days are not touched |
| Removed event | T12, T13: removed Silver rows lower one Gold row and remove another |
| Failed check | T16 to T18: a failing quality gate leaves Gold unchanged, then a recovery run |
| A day that is gone | T19, T20: a run for all dates removes a day that no longer exists in Silver |
| Bad parameter | T21 to T23: an invalid `source_date`, `test_schema` or `run_id` stops the notebook in its parameter step. T24: nothing was written |
| Production safety | T24: no production table changed |

## T1. Setup: test schema and empty tables

The test schema is dropped and created again, so every test run starts clean.

`CREATE TABLE ... LIKE` makes an empty table with the same columns as another table. The Silver test table is an empty copy of the real `silver.pull_request_events`, and the audit test table is an empty copy of the real audit table.

The Gold test table is not created here. The notebook under test creates it, which also tests its `CREATE TABLE` statement.

In [0]:
TEST_SCHEMA = "test_gold_g1"

TEST_SILVER = f"github_lakehouse.{TEST_SCHEMA}.pull_request_events"
TEST_GOLD = f"github_lakehouse.{TEST_SCHEMA}.repository_delivery_metrics"
TEST_STAGING = f"github_lakehouse.{TEST_SCHEMA}.repository_delivery_metrics_staging"
TEST_AUDIT = f"github_lakehouse.{TEST_SCHEMA}.gold_pipeline_runs"
TEST_COPY = f"github_lakehouse.{TEST_SCHEMA}.gold_rows_after_run_1"

PROD_SILVER = "github_lakehouse.silver.pull_request_events"
PROD_AUDIT = "github_lakehouse.audit.gold_pipeline_runs"
PROD_GOLD = "github_lakehouse.gold.repository_delivery_metrics"
PROD_STAGING = "github_lakehouse.gold_staging.repository_delivery_metrics"

# Every time in this test is UTC.
spark.conf.set("spark.sql.session.timeZone", "UTC")

# Every check section writes its number of failed checks into this dictionary.
# The cell "Result of the whole test" reads it at the end.
section_results = {}

if not spark.catalog.tableExists(PROD_AUDIT):
    raise RuntimeError(f"{PROD_AUDIT} does not exist. Run 08_gold_audit_table first.")

if not spark.catalog.tableExists(PROD_SILVER):
    raise RuntimeError(f"{PROD_SILVER} does not exist. Run the Silver notebook 02_pull_request_events first.")

# Only a schema whose name starts with test_ is ever dropped.
if not TEST_SCHEMA.startswith("test_"):
    raise RuntimeError("Refusing to drop a schema that is not a test schema.")

spark.sql(f"DROP SCHEMA IF EXISTS github_lakehouse.{TEST_SCHEMA} CASCADE")
spark.sql(f"CREATE SCHEMA github_lakehouse.{TEST_SCHEMA}")

spark.sql(f"CREATE TABLE {TEST_SILVER} LIKE {PROD_SILVER}")
spark.sql(f"CREATE TABLE {TEST_AUDIT} LIKE {PROD_AUDIT}")

print("Test schema ready:", TEST_SCHEMA)

## T2. Remember the state of the production tables

The Delta version of a table goes up with every write. The versions are read now and again at the end. If they are the same, the test did not write to production.

The production Gold table and its staging copy may not exist yet. The value `-1` means: the table does not exist.

In [0]:
prod_audit_version_before = spark.sql(
    f"DESCRIBE HISTORY {PROD_AUDIT} LIMIT 1"
).collect()[0]["version"]

if spark.catalog.tableExists(PROD_GOLD):
    prod_gold_version_before = spark.sql(
        f"DESCRIBE HISTORY {PROD_GOLD} LIMIT 1"
    ).collect()[0]["version"]
else:
    prod_gold_version_before = -1

if spark.catalog.tableExists(PROD_STAGING):
    prod_staging_version_before = spark.sql(
        f"DESCRIBE HISTORY {PROD_STAGING} LIMIT 1"
    ).collect()[0]["version"]
else:
    prod_staging_version_before = -1

print("Production audit version  :", prod_audit_version_before)
print("Production Gold version   :", prod_gold_version_before)
print("Production staging version:", prod_staging_version_before)

## T3. Run on the empty Silver table

In [0]:
%run ../01_repository_delivery_metrics $test_schema="test_gold_g1" $source_date="" $run_id="test-dry-run"

## T4. Checks: no input

An empty Silver table has no days to rebuild. The run must end as `NO_INPUT`, publish nothing and still write its audit row.

In every check table, `<=>` compares two values and also treats NULL and NULL as equal.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 the notebook read the test Silver table' AS check_name,
               '{SILVER_TABLE}' AS actual, '{TEST_SILVER}' AS expected
        UNION ALL
        SELECT '02 the notebook wrote the test Gold table', '{GOLD_TABLE}', '{TEST_GOLD}'
        UNION ALL
        SELECT '03 the notebook wrote the test staging table', '{STAGING_TABLE}', '{TEST_STAGING}'
        UNION ALL
        SELECT '04 the notebook wrote the test audit table', '{AUDIT_TABLE}', '{TEST_AUDIT}'
        UNION ALL
        SELECT '05 Gold rows', CAST(COUNT(*) AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '06 audit rows', CAST(COUNT(*) AS STRING), '1' FROM {TEST_AUDIT}
        UNION ALL
        SELECT '07 audit status', MAX(status), 'NO_INPUT'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '08 audit pipeline_name', MAX(pipeline_name), 'gold_repository_delivery_metrics'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '09 audit gold_table', MAX(gold_table), '{TEST_GOLD}'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '10 audit processing_scope', MAX(processing_scope), 'ALL_DATES'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '11 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '12 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '13 audit number of affected dates', CAST(MAX(size(affected_dates)) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '14 audit Silver rows read', CAST(MAX(source_rows_read['pull_request_events']) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '15 audit repository_days', CAST(MAX(quality_metrics['repository_days']) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T4"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T4: {failed_count} check(s) failed. See the table above.")
print("T4: ALL CHECKS PASSED")

## T5. Load the hand-made Silver rows

31 pull request events on three days. Only the columns that G1 reads are filled, the others stay NULL.

**Repository 100 on 1 February: 15 events, one for every rule**

| Event | Action | Pull request | What it tests |
|---|---|---|---|
| ...01 | `opened` | 1 | A pull request is opened |
| ...02 | `closed` | 1 | and merged the same day, 2 hours after its creation. Its latest event, so its snapshot |
| ...03 | `closed` | 2 | Closed without a merge, 3 hours after its creation |
| ...04 | `reopened` | 2 | then reopened |
| ...05 | `closed` | 2 | and closed again, 7 hours after its **original** creation. Three events, one snapshot |
| ...06 | `closed` | 3 | A merge time, but the merged flag is unknown: not in the merge share and not in the merge hours. Commits and deleted lines are unknown |
| ...07 | `opened` | 4 | An `opened` event that already says merged and carries a close time: it counts only as opened |
| ...08 | `closed` | 5 | Merged, but without a merge time: in the merge share, not in the merge hours. No size at all |
| ...09 | `closed` | 6 | Merged and closed before it was created: no duration |
| ...10 | `closed` | 7 | Closed without a creation time: no duration |
| ...11 | none | 8 | An unknown action. It is the latest event of pull request 8, so it is its snapshot: a snapshot does not need a known action |
| ...12 | `edited` | 8 | A known action that is none of the three. Earlier than ...11, so its sizes are not used |
| ...13 | `opened` | number 77 | No `pr_id`: the pull request is identified by its number |
| ...14 | `closed` | number 77 | The same pull request, merged after 20 minutes. Same second as ...13, but the larger `event_id` |
| ...15 | `opened` | none | No ID and no number, and no repository name. It is the latest event of the day. Its sizes must not be used |

**The other rows**

| Repository | Day | Events | What it tests |
|---|---|---|---|
| 200 | 1 Feb | 2 | A day with only `opened` events: no share and no duration. 0 added lines is a value, not NULL. One of the two pull requests has no `pr_id` and the number 77, the same number as in repository 100 |
| 200 | none | 1 | An event without event time and date: it cannot be placed on a day |
| 200 | 2 Feb | 1 | The same repository on a second day, with a new name. Pull request 20 is merged after 24 hours. It has a snapshot on both days |
| 400 | 2 Feb | 2 | No name and no size. One close without a merge: the share is 0, not NULL |
| 300 | 5 Feb | 5 | Four merged after 1, 2, 3 and 10 hours, one closed without a merge after 20 hours: the median of an even and of an odd number of values |
| 310 | 5 Feb | 1 | Created 1 December 2023 12:00, merged 5 February 2024 12:00, closed at 12:15. The event itself is at 12:30, and that must not count |
| 320 | 5 Feb | 3 | A share, hours and sizes that must be rounded |
| 600 | 5 Feb | 1 | An event without a pull request key: the row exists, with 0 snapshots |

In [0]:
spark.sql("INSERT INTO " + TEST_SILVER + """
    (event_id, event_timestamp, event_date, repo_id, repo_name,
     pr_action, pr_id, pr_number, pr_created_at, pr_closed_at, pr_merged_at, pr_merged,
     pr_commits, pr_additions, pr_deletions, pr_changed_files, _source_date)
VALUES
    -- Repository 100 on 1 February
    ('36000000001', TIMESTAMP'2024-02-01 08:00:00', DATE'2024-02-01', 100, 'org/alpha',        'opened',   1,    1,    TIMESTAMP'2024-02-01 08:00:00', NULL,                           NULL,                           false, 1,    10,   0,    1,    DATE'2024-02-01'),
    ('36000000002', TIMESTAMP'2024-02-01 10:00:00', DATE'2024-02-01', 100, 'org/alpha',        'closed',   1,    1,    TIMESTAMP'2024-02-01 08:00:00', TIMESTAMP'2024-02-01 10:00:00', TIMESTAMP'2024-02-01 10:00:00', true,  2,    30,   5,    2,    DATE'2024-02-01'),
    ('36000000003', TIMESTAMP'2024-02-01 09:00:00', DATE'2024-02-01', 100, 'org/alpha',        'closed',   2,    2,    TIMESTAMP'2024-02-01 06:00:00', TIMESTAMP'2024-02-01 09:00:00', NULL,                           false, 4,    100,  50,   3,    DATE'2024-02-01'),
    ('36000000004', TIMESTAMP'2024-02-01 11:00:00', DATE'2024-02-01', 100, 'org/alpha',        'reopened', 2,    2,    TIMESTAMP'2024-02-01 06:00:00', NULL,                           NULL,                           false, 4,    100,  50,   3,    DATE'2024-02-01'),
    ('36000000005', TIMESTAMP'2024-02-01 13:00:00', DATE'2024-02-01', 100, 'org/alpha',        'closed',   2,    2,    TIMESTAMP'2024-02-01 06:00:00', TIMESTAMP'2024-02-01 13:00:00', NULL,                           false, 5,    120,  60,   4,    DATE'2024-02-01'),
    ('36000000006', TIMESTAMP'2024-02-01 14:00:00', DATE'2024-02-01', 100, 'org/alpha',        'closed',   3,    3,    TIMESTAMP'2024-02-01 12:00:00', TIMESTAMP'2024-02-01 14:00:00', TIMESTAMP'2024-02-01 14:00:00', NULL,  NULL, 40,   NULL, 2,    DATE'2024-02-01'),
    ('36000000007', TIMESTAMP'2024-02-01 15:00:00', DATE'2024-02-01', 100, 'org/alpha',        'opened',   4,    4,    TIMESTAMP'2024-02-01 15:00:00', TIMESTAMP'2024-02-01 15:00:30', TIMESTAMP'2024-02-01 15:00:30', true,  1,    1,    1,    NULL, DATE'2024-02-01'),
    ('36000000008', TIMESTAMP'2024-02-01 16:00:00', DATE'2024-02-01', 100, 'org/alpha',        'closed',   5,    5,    TIMESTAMP'2024-02-01 15:00:00', TIMESTAMP'2024-02-01 16:00:00', NULL,                           true,  NULL, NULL, NULL, NULL, DATE'2024-02-01'),
    ('36000000009', TIMESTAMP'2024-02-01 17:00:00', DATE'2024-02-01', 100, 'org/alpha',        'closed',   6,    6,    TIMESTAMP'2024-02-01 17:00:00', TIMESTAMP'2024-02-01 16:00:00', TIMESTAMP'2024-02-01 16:00:00', true,  3,    9,    NULL, 9,    DATE'2024-02-01'),
    ('36000000010', TIMESTAMP'2024-02-01 17:30:00', DATE'2024-02-01', 100, 'org/alpha',        'closed',   7,    7,    NULL,                           TIMESTAMP'2024-02-01 17:30:00', NULL,                           false, 2,    2,    2,    NULL, DATE'2024-02-01'),
    ('36000000011', TIMESTAMP'2024-02-01 18:30:00', DATE'2024-02-01', 100, 'org/alpha',        NULL,       8,    8,    TIMESTAMP'2024-02-01 18:00:00', NULL,                           NULL,                           NULL,  1,    21,   4,    NULL, DATE'2024-02-01'),
    ('36000000012', TIMESTAMP'2024-02-01 18:00:00', DATE'2024-02-01', 100, 'org/alpha',        'edited',   8,    8,    TIMESTAMP'2024-02-01 18:00:00', NULL,                           NULL,                           false, 6,    60,   6,    6,    DATE'2024-02-01'),
    ('36000000013', TIMESTAMP'2024-02-01 19:20:00', DATE'2024-02-01', 100, 'org/alpha-old',    'opened',   NULL, 77,   TIMESTAMP'2024-02-01 19:00:00', NULL,                           NULL,                           false, 1,    5,    0,    1,    DATE'2024-02-01'),
    ('36000000014', TIMESTAMP'2024-02-01 19:20:00', DATE'2024-02-01', 100, 'org/alpha-new',    'closed',   NULL, 77,   TIMESTAMP'2024-02-01 19:00:00', TIMESTAMP'2024-02-01 19:20:00', TIMESTAMP'2024-02-01 19:20:00', true,  2,    7,    1,    1,    DATE'2024-02-01'),
    ('36000000015', TIMESTAMP'2024-02-01 20:00:00', DATE'2024-02-01', 100, NULL,               'opened',   NULL, NULL, TIMESTAMP'2024-02-01 20:00:00', NULL,                           NULL,                           false, 9,    999,  999,  99,   DATE'2024-02-01'),

    -- Repository 200 on 1 February: only opened events
    ('36000000016', TIMESTAMP'2024-02-01 09:00:00', DATE'2024-02-01', 200, 'org/beta',         'opened',   20,   1,    TIMESTAMP'2024-02-01 09:00:00', NULL,                           NULL,                           false, 1,    0,    0,    1,    DATE'2024-02-01'),
    ('36000000017', TIMESTAMP'2024-02-01 09:30:00', DATE'2024-02-01', 200, 'org/beta',         'opened',   NULL, 77,   TIMESTAMP'2024-02-01 09:30:00', NULL,                           NULL,                           false, 3,    10,   0,    2,    DATE'2024-02-01'),

    -- Repository 200: an event without event time and event date
    ('36000000018', NULL,                           NULL,             200, 'org/beta',         'opened',   23,   4,    TIMESTAMP'2024-02-01 10:00:00', NULL,                           NULL,                           false, 1,    1,    1,    1,    DATE'2024-02-01'),

    -- Repository 200 on 2 February: a new name, merged after 24 hours
    ('36000000019', TIMESTAMP'2024-02-02 09:00:00', DATE'2024-02-02', 200, 'org/beta-renamed', 'closed',   20,   1,    TIMESTAMP'2024-02-01 09:00:00', TIMESTAMP'2024-02-02 09:00:00', TIMESTAMP'2024-02-02 09:00:00', true,  1,    0,    0,    1,    DATE'2024-02-02'),

    -- Repository 400 on 2 February: no name, no size
    ('36000000020', TIMESTAMP'2024-02-02 10:00:00', DATE'2024-02-02', 400, NULL,               'opened',   40,   1,    TIMESTAMP'2024-02-02 10:00:00', NULL,                           NULL,                           false, NULL, NULL, NULL, NULL, DATE'2024-02-02'),
    ('36000000021', TIMESTAMP'2024-02-02 10:30:00', DATE'2024-02-02', 400, NULL,               'closed',   41,   2,    TIMESTAMP'2024-02-02 08:00:00', TIMESTAMP'2024-02-02 10:30:00', NULL,                           false, NULL, NULL, NULL, NULL, DATE'2024-02-02'),

    -- Repository 300 on 5 February: merged after 1, 2, 3 and 10 hours, one closed after 20 hours
    ('36000000022', TIMESTAMP'2024-02-05 01:00:00', DATE'2024-02-05', 300, 'org/gamma',        'closed',   30,   1,    TIMESTAMP'2024-02-05 00:00:00', TIMESTAMP'2024-02-05 01:00:00', TIMESTAMP'2024-02-05 01:00:00', true,  1,    10,   1,    1,    DATE'2024-02-05'),
    ('36000000023', TIMESTAMP'2024-02-05 02:00:00', DATE'2024-02-05', 300, 'org/gamma',        'closed',   31,   2,    TIMESTAMP'2024-02-05 00:00:00', TIMESTAMP'2024-02-05 02:00:00', TIMESTAMP'2024-02-05 02:00:00', true,  1,    20,   2,    1,    DATE'2024-02-05'),
    ('36000000024', TIMESTAMP'2024-02-05 03:00:00', DATE'2024-02-05', 300, 'org/gamma',        'closed',   32,   3,    TIMESTAMP'2024-02-05 00:00:00', TIMESTAMP'2024-02-05 03:00:00', TIMESTAMP'2024-02-05 03:00:00', true,  1,    30,   3,    1,    DATE'2024-02-05'),
    ('36000000025', TIMESTAMP'2024-02-05 10:00:00', DATE'2024-02-05', 300, 'org/gamma',        'closed',   33,   4,    TIMESTAMP'2024-02-05 00:00:00', TIMESTAMP'2024-02-05 10:00:00', TIMESTAMP'2024-02-05 10:00:00', true,  1,    40,   4,    2,    DATE'2024-02-05'),
    ('36000000026', TIMESTAMP'2024-02-05 00:00:00', DATE'2024-02-05', 300, 'org/gamma',        'closed',   34,   5,    TIMESTAMP'2024-02-04 04:00:00', TIMESTAMP'2024-02-05 00:00:00', NULL,                           false, 2,    50,   5,    2,    DATE'2024-02-05'),

    -- Repository 310 on 5 February: a pull request from 1 December 2023
    ('36000000027', TIMESTAMP'2024-02-05 12:30:00', DATE'2024-02-05', 310, 'org/delta',        'closed',   50,   9,    TIMESTAMP'2023-12-01 12:00:00', TIMESTAMP'2024-02-05 12:15:00', TIMESTAMP'2024-02-05 12:00:00', true,  12,   3456, 789,  25,   DATE'2024-02-05'),

    -- Repository 320 on 5 February: values that must be rounded
    ('36000000028', TIMESTAMP'2024-02-05 06:20:00', DATE'2024-02-05', 320, 'org/epsilon',      'closed',   60,   1,    TIMESTAMP'2024-02-05 06:00:00', TIMESTAMP'2024-02-05 06:20:00', TIMESTAMP'2024-02-05 06:20:00', true,  1,    1,    NULL, 3,    DATE'2024-02-05'),
    ('36000000029', TIMESTAMP'2024-02-05 08:00:00', DATE'2024-02-05', 320, 'org/epsilon',      'closed',   61,   2,    TIMESTAMP'2024-02-05 07:00:00', TIMESTAMP'2024-02-05 08:00:00', TIMESTAMP'2024-02-05 08:00:00', true,  2,    1,    NULL, 4,    DATE'2024-02-05'),
    ('36000000030', TIMESTAMP'2024-02-05 09:30:00', DATE'2024-02-05', 320, 'org/epsilon',      'closed',   62,   3,    TIMESTAMP'2024-02-05 09:00:00', TIMESTAMP'2024-02-05 09:30:00', NULL,                           false, 2,    2,    NULL, 3,    DATE'2024-02-05'),

    -- Repository 600 on 5 February: an event without a pull request key
    ('36000000031', TIMESTAMP'2024-02-05 15:00:00', DATE'2024-02-05', 600, 'org/eta',          'opened',   NULL, NULL, TIMESTAMP'2024-02-05 15:00:00', NULL,                           NULL,                           false, 5,    50,   5,    5,    DATE'2024-02-05')
""")

print("Silver test rows:", spark.table(TEST_SILVER).count())

### A two-line summary of every Gold row

The checks below compare whole Gold rows. This view writes the value columns of a row into two readable lines, for example:

`events=2 known=2 opened=2 closed=0 reopened=0 other=0 eligible_closed=0 merged=0 share=NULL merged_dur=0 avg_merge=NULL p50_merge=NULL closed_dur=0 avg_close=NULL p50_close=NULL`

`snapshots=2 commits_n=2 avg_commits=2.0 additions_n=2 avg_additions=5.0 deletions_n=2 avg_deletions=0.0 files_n=2 avg_files=1.5 name=org/beta`

The first line holds the numbers that count events. The second line holds the numbers that count snapshots, and the name.

The view is not a copy. It always reads the Gold table as it is at that moment.

In [0]:
gold_summary = spark.sql(f"""
    SELECT
        repo_id,
        activity_date,
        concat(
            'events=',           CAST(pr_event_count AS STRING),
            ' known=',           CAST(pr_action_known_rows AS STRING),
            ' opened=',          CAST(pr_opened_actions AS STRING),
            ' closed=',          CAST(pr_closed_actions AS STRING),
            ' reopened=',        CAST(pr_reopened_actions AS STRING),
            ' other=',           CAST(pr_other_action_rows AS STRING),
            ' eligible_closed=', CAST(eligible_closed_rows AS STRING),
            ' merged=',          CAST(merged_closed_rows AS STRING),
            ' share=',           COALESCE(CAST(merge_share_pct AS STRING), 'NULL'),
            ' merged_dur=',      CAST(merged_duration_eligible_rows AS STRING),
            ' avg_merge=',       COALESCE(CAST(avg_merge_hours AS STRING), 'NULL'),
            ' p50_merge=',       COALESCE(CAST(p50_merge_hours AS STRING), 'NULL'),
            ' closed_dur=',      CAST(closed_duration_eligible_rows AS STRING),
            ' avg_close=',       COALESCE(CAST(avg_close_hours AS STRING), 'NULL'),
            ' p50_close=',       COALESCE(CAST(p50_close_hours AS STRING), 'NULL')
        ) AS event_summary,
        concat(
            'snapshots=',        CAST(pr_size_snapshot_rows AS STRING),
            ' commits_n=',       CAST(pr_commits_eligible_rows AS STRING),
            ' avg_commits=',     COALESCE(CAST(avg_pr_commits AS STRING), 'NULL'),
            ' additions_n=',     CAST(pr_additions_eligible_rows AS STRING),
            ' avg_additions=',   COALESCE(CAST(avg_pr_additions AS STRING), 'NULL'),
            ' deletions_n=',     CAST(pr_deletions_eligible_rows AS STRING),
            ' avg_deletions=',   COALESCE(CAST(avg_pr_deletions AS STRING), 'NULL'),
            ' files_n=',         CAST(pr_changed_files_eligible_rows AS STRING),
            ' avg_files=',       COALESCE(CAST(avg_pr_changed_files AS STRING), 'NULL'),
            ' name=',            COALESCE(repo_name, 'NULL')
        ) AS size_summary
    FROM {TEST_GOLD}
""")
gold_summary.createOrReplaceTempView("gold_summary")

## T6. Run 1: all dates

In [0]:
%run ../01_repository_delivery_metrics $test_schema="test_gold_g1" $source_date="" $run_id="test-run1"

## T7. Checks of run 1: schema and exact values

Worked out by hand.

**The run.** 30 of the 31 rows have an event date, on 3 days. The row without a date is left out and counted. The 30 rows belong to 8 repository-days.

**Repository 100 on 1 February: the events.**

- 15 events. 14 have a known action: 4 opened (...01, ...07, ...13, ...15), 8 closed (...02, ...03, ...05, ...06, ...08, ...09, ...10, ...14), 1 reopened, 1 other.
- Merge share: 7 of the 8 close events have a known merged flag, all but ...06. 4 of them are merged: ...02, ...08, ...09 and ...14. 100 × 4 ÷ 7 = 57.1429, rounded 57.14. The `opened` event ...07 says merged too, but it is not a close event.
- Merge hours: 2 of the 4 merged closes are eligible: 2 hours (...02) and 20 minutes, which is 0.3333 hours (...14). ...08 has no merge time, and ...09 was merged before its creation. The close event ...06 carries a merge time, but its merged flag is unknown, so it is not used. Average and median: (2 + 0.3333) ÷ 2 = 1.1667, rounded 1.17.
- Close hours: 6 of the 8 closes are eligible: 2, 3, 7, 2, 1 and 0.3333 hours. ...09 was closed before its creation, and ...10 has no creation time. Average: 15.3333 ÷ 6 = 2.5556, rounded 2.56. Median of 0.3333, 1, 2, 2, 3 and 7: the average of the two middle values, 2.

**Repository 100 on 1 February: the snapshots.**

- 14 events have a pull request key, all but ...15. They belong to 9 pull requests: the IDs 1 to 8 and the number 77.
- The snapshot of each pull request is its latest event of the day.

| Snapshot | Commits | Added lines | Deleted lines | Files |
|---|---|---|---|---|
| ...02 | 2 | 30 | 5 | 2 |
| ...05 | 5 | 120 | 60 | 4 |
| ...06 | unknown | 40 | unknown | 2 |
| ...07 | 1 | 1 | 1 | unknown |
| ...08 | unknown | unknown | unknown | unknown |
| ...09 | 3 | 9 | unknown | 9 |
| ...10 | 2 | 2 | 2 | unknown |
| ...11 | 1 | 21 | 4 | unknown |
| ...14 | 2 | 7 | 1 | 1 |
| **Known values** | **7** | **8** | **6** | **5** |
| **Sum** | **16** | **230** | **73** | **18** |
| **Average** | **2.29** | **28.75** | **12.17** | **3.6** |

- The snapshot of pull request 8 is ...11, although its action is unknown. A snapshot is the latest event of the day, whatever its action.
- The sizes on the other events are not used: ...01, ...03, ...04, ...12 and ...13 are not the latest event of their pull request, and ...15 has no pull request key.
- Label: the latest event (...15) has no name, so it is skipped. The next two share the same second. The larger `event_id` is ...14, so the label is `org/alpha-new`.

**The other rows.**

- Repository 200 on 1 February: 2 opened events, nothing closed. The share and the four durations are NULL, and their counts are 0. 2 snapshots: commits (1 + 3) ÷ 2 = 2, added lines (0 + 10) ÷ 2 = 5, deleted lines (0 + 0) ÷ 2 = 0, files (1 + 2) ÷ 2 = 1.5. The average of the deleted lines is 0 and not NULL, because both zeros are known values. One of the two pull requests is identified by the number 77. Repository 100 has a pull request with the same number on the same day. They are two different pull requests, because a number counts only inside its repository.
- Repository 200 on 2 February: one merge after 24 hours. The share is 100. Its label is the new name.
- Repository 400 on 2 February: 2 events without a name. One close without a merge, after 2.5 hours: the share is 0 and there is no merge duration. The 2 snapshots have no known size, so the four size counts are 0 and the four averages are NULL. The label is NULL.
- Repository 300 on 5 February: 5 closes, 4 of them merged: share 80. Merge hours 1, 2, 3 and 10: average 16 ÷ 4 = 4, median (2 + 3) ÷ 2 = 2.5. Close hours 1, 2, 3, 10 and 20: average 36 ÷ 5 = 7.2, median the middle value, 3. Sizes: commits 6 ÷ 5 = 1.2, added lines 150 ÷ 5 = 30, deleted lines 15 ÷ 5 = 3, files 7 ÷ 5 = 1.4.
- Repository 310 on 5 February: 1 December 12:00 to 5 February 12:00 is 30 + 31 + 5 = 66 days, which is 1,584 hours to the merge. The close is 15 minutes later: 1,584.25 hours. Both are counted on 5 February, the day of the close event. The time of the event itself is not used.
- Repository 320 on 5 February: 2 of 3 closes are merged: 66.6667, rounded 66.67. Merge hours 0.3333 and 1: average and median 0.6667, rounded 0.67. Close hours 0.3333, 1 and 0.5: average 1.8333 ÷ 3 = 0.6111, rounded 0.61, median 0.5. Commits (1 + 2 + 2) ÷ 3 = 1.6667, rounded 1.67. Added lines (1 + 1 + 2) ÷ 3 = 1.3333, rounded 1.33. No deleted lines are known. Files (3 + 4 + 3) ÷ 3 = 3.3333, rounded 3.33.
- Repository 600 on 5 February: one event without a pull request key. The row exists. It has 0 snapshots, so the five size counts are 0 and the four averages are NULL. The sizes on the event are not used.

**The totals of the run.** 30 events, 29 with a known action: 8 opened, 19 closed, 1 reopened, 1 other. 18 closes with a known merged flag, 12 of them merged. 10 merge durations and 17 close durations. 23 snapshots: 19 with commits, 20 with added lines, 15 with deleted lines, 17 with files.

In [0]:
actual_schema = spark.table(TEST_GOLD).schema.simpleString()
staging_schema = spark.table(TEST_STAGING).schema.simpleString()

expected_schema = (
    "struct<repo_id:bigint,activity_date:date,repo_name:string,"
    "pr_event_count:bigint,pr_action_known_rows:bigint,pr_opened_actions:bigint,"
    "pr_closed_actions:bigint,pr_reopened_actions:bigint,pr_other_action_rows:bigint,"
    "eligible_closed_rows:bigint,merged_closed_rows:bigint,merge_share_pct:double,"
    "merged_duration_eligible_rows:bigint,avg_merge_hours:double,p50_merge_hours:double,"
    "closed_duration_eligible_rows:bigint,avg_close_hours:double,p50_close_hours:double,"
    "pr_size_snapshot_rows:bigint,"
    "pr_commits_eligible_rows:bigint,avg_pr_commits:double,"
    "pr_additions_eligible_rows:bigint,avg_pr_additions:double,"
    "pr_deletions_eligible_rows:bigint,avg_pr_deletions:double,"
    "pr_changed_files_eligible_rows:bigint,avg_pr_changed_files:double,"
    "_gold_run_id:string,_gold_updated_at:timestamp>"
)

print("Gold    :", actual_schema)
print("Staging :", staging_schema)
print("Expected:", expected_schema)

# The three texts are long. The check table below gets a short text for each comparison.
if actual_schema == expected_schema:
    gold_schema_result = "as in the contract"
else:
    gold_schema_result = "DIFFERENT, see the lines above"

if staging_schema == expected_schema:
    staging_schema_result = "as in the contract"
else:
    staging_schema_result = "DIFFERENT, see the lines above"

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 columns and types of the Gold table' AS check_name,
               '{gold_schema_result}' AS actual, 'as in the contract' AS expected
        UNION ALL
        SELECT '02 columns and types of the staging table', '{staging_schema_result}', 'as in the contract'
        UNION ALL
        SELECT '03 Gold rows', CAST(COUNT(*) AS STRING), '8' FROM {TEST_GOLD}
        UNION ALL
        SELECT '04 different days', CAST(COUNT(DISTINCT activity_date) AS STRING), '3' FROM {TEST_GOLD}
        UNION ALL
        SELECT '05 rows written by this run', CAST(count_if(_gold_run_id = 'test-run1' AND _gold_updated_at IS NOT NULL) AS STRING), '8' FROM {TEST_GOLD}
        UNION ALL
        SELECT '06 sum of pr_event_count', CAST(SUM(pr_event_count) AS STRING), '30' FROM {TEST_GOLD}
        UNION ALL
        SELECT '07 repository 100 on 1 February: events', MAX(event_summary), 'events=15 known=14 opened=4 closed=8 reopened=1 other=1 eligible_closed=7 merged=4 share=57.14 merged_dur=2 avg_merge=1.17 p50_merge=1.17 closed_dur=6 avg_close=2.56 p50_close=2.0'
        FROM gold_summary WHERE repo_id = 100 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '08 repository 100 on 1 February: sizes', MAX(size_summary), 'snapshots=9 commits_n=7 avg_commits=2.29 additions_n=8 avg_additions=28.75 deletions_n=6 avg_deletions=12.17 files_n=5 avg_files=3.6 name=org/alpha-new'
        FROM gold_summary WHERE repo_id = 100 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '09 repository 200 on 1 February (only opened): events', MAX(event_summary), 'events=2 known=2 opened=2 closed=0 reopened=0 other=0 eligible_closed=0 merged=0 share=NULL merged_dur=0 avg_merge=NULL p50_merge=NULL closed_dur=0 avg_close=NULL p50_close=NULL'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '10 repository 200 on 1 February (only opened): sizes', MAX(size_summary), 'snapshots=2 commits_n=2 avg_commits=2.0 additions_n=2 avg_additions=5.0 deletions_n=2 avg_deletions=0.0 files_n=2 avg_files=1.5 name=org/beta'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '11 repository 200 on 2 February (new name): events', MAX(event_summary), 'events=1 known=1 opened=0 closed=1 reopened=0 other=0 eligible_closed=1 merged=1 share=100.0 merged_dur=1 avg_merge=24.0 p50_merge=24.0 closed_dur=1 avg_close=24.0 p50_close=24.0'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '12 repository 200 on 2 February (new name): sizes', MAX(size_summary), 'snapshots=1 commits_n=1 avg_commits=1.0 additions_n=1 avg_additions=0.0 deletions_n=1 avg_deletions=0.0 files_n=1 avg_files=1.0 name=org/beta-renamed'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '13 repository 400 on 2 February (no name, no size): events', MAX(event_summary), 'events=2 known=2 opened=1 closed=1 reopened=0 other=0 eligible_closed=1 merged=0 share=0.0 merged_dur=0 avg_merge=NULL p50_merge=NULL closed_dur=1 avg_close=2.5 p50_close=2.5'
        FROM gold_summary WHERE repo_id = 400 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '14 repository 400 on 2 February (no name, no size): sizes', MAX(size_summary), 'snapshots=2 commits_n=0 avg_commits=NULL additions_n=0 avg_additions=NULL deletions_n=0 avg_deletions=NULL files_n=0 avg_files=NULL name=NULL'
        FROM gold_summary WHERE repo_id = 400 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '15 repository 300 on 5 February (medians): events', MAX(event_summary), 'events=5 known=5 opened=0 closed=5 reopened=0 other=0 eligible_closed=5 merged=4 share=80.0 merged_dur=4 avg_merge=4.0 p50_merge=2.5 closed_dur=5 avg_close=7.2 p50_close=3.0'
        FROM gold_summary WHERE repo_id = 300 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '16 repository 300 on 5 February (medians): sizes', MAX(size_summary), 'snapshots=5 commits_n=5 avg_commits=1.2 additions_n=5 avg_additions=30.0 deletions_n=5 avg_deletions=3.0 files_n=5 avg_files=1.4 name=org/gamma'
        FROM gold_summary WHERE repo_id = 300 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '17 repository 310 on 5 February (1,584 hours): events', MAX(event_summary), 'events=1 known=1 opened=0 closed=1 reopened=0 other=0 eligible_closed=1 merged=1 share=100.0 merged_dur=1 avg_merge=1584.0 p50_merge=1584.0 closed_dur=1 avg_close=1584.25 p50_close=1584.25'
        FROM gold_summary WHERE repo_id = 310 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '18 repository 310 on 5 February (1,584 hours): sizes', MAX(size_summary), 'snapshots=1 commits_n=1 avg_commits=12.0 additions_n=1 avg_additions=3456.0 deletions_n=1 avg_deletions=789.0 files_n=1 avg_files=25.0 name=org/delta'
        FROM gold_summary WHERE repo_id = 310 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '19 repository 320 on 5 February (rounding): events', MAX(event_summary), 'events=3 known=3 opened=0 closed=3 reopened=0 other=0 eligible_closed=3 merged=2 share=66.67 merged_dur=2 avg_merge=0.67 p50_merge=0.67 closed_dur=3 avg_close=0.61 p50_close=0.5'
        FROM gold_summary WHERE repo_id = 320 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '20 repository 320 on 5 February (rounding): sizes', MAX(size_summary), 'snapshots=3 commits_n=3 avg_commits=1.67 additions_n=3 avg_additions=1.33 deletions_n=0 avg_deletions=NULL files_n=3 avg_files=3.33 name=org/epsilon'
        FROM gold_summary WHERE repo_id = 320 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '21 repository 600 on 5 February (no snapshot): events', MAX(event_summary), 'events=1 known=1 opened=1 closed=0 reopened=0 other=0 eligible_closed=0 merged=0 share=NULL merged_dur=0 avg_merge=NULL p50_merge=NULL closed_dur=0 avg_close=NULL p50_close=NULL'
        FROM gold_summary WHERE repo_id = 600 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '22 repository 600 on 5 February (no snapshot): sizes', MAX(size_summary), 'snapshots=0 commits_n=0 avg_commits=NULL additions_n=0 avg_additions=NULL deletions_n=0 avg_deletions=NULL files_n=0 avg_files=NULL name=org/eta'
        FROM gold_summary WHERE repo_id = 600 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '23 audit status', MAX(status), 'SUCCEEDED'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '24 audit processing_scope', MAX(processing_scope), 'ALL_DATES'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '25 audit source_date', CAST(MAX(source_date) AS STRING), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '26 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-01, 2024-02-02, 2024-02-05]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '27 audit Silver rows read', CAST(MAX(source_rows_read['pull_request_events']) AS STRING), '30'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '28 audit rows without an event date', CAST(MAX(rows_excluded['missing_event_date']) AS STRING), '1'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '29 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '8'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '30 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '31 audit failed_checks', MAX(failed_checks), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '32 audit error_message', MAX(error_message), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '33 audit has a table version', CAST(MAX(gold_table_version) IS NOT NULL AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '34 audit pipeline_name', MAX(pipeline_name), 'gold_repository_delivery_metrics'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '35 audit gold_table', MAX(gold_table), '{TEST_GOLD}'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '36 audit run times are in order', CAST(MAX(run_finished_at >= run_started_at) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '37 audit quality_metrics', MAX(concat('repository_days=', CAST(quality_metrics['repository_days'] AS STRING), ' events=', CAST(quality_metrics['pr_event_count'] AS STRING), ' known=', CAST(quality_metrics['pr_action_known_rows'] AS STRING), ' opened=', CAST(quality_metrics['pr_opened_actions'] AS STRING), ' closed=', CAST(quality_metrics['pr_closed_actions'] AS STRING), ' reopened=', CAST(quality_metrics['pr_reopened_actions'] AS STRING), ' other=', CAST(quality_metrics['pr_other_action_rows'] AS STRING), ' eligible_closed=', CAST(quality_metrics['eligible_closed_rows'] AS STRING), ' merged=', CAST(quality_metrics['merged_closed_rows'] AS STRING), ' merged_dur=', CAST(quality_metrics['merged_duration_eligible_rows'] AS STRING), ' closed_dur=', CAST(quality_metrics['closed_duration_eligible_rows'] AS STRING), ' snapshots=', CAST(quality_metrics['pr_size_snapshot_rows'] AS STRING), ' commits_n=', CAST(quality_metrics['pr_commits_eligible_rows'] AS STRING), ' additions_n=', CAST(quality_metrics['pr_additions_eligible_rows'] AS STRING), ' deletions_n=', CAST(quality_metrics['pr_deletions_eligible_rows'] AS STRING), ' files_n=', CAST(quality_metrics['pr_changed_files_eligible_rows'] AS STRING))),
               'repository_days=8 events=30 known=29 opened=8 closed=19 reopened=1 other=1 eligible_closed=18 merged=12 merged_dur=10 closed_dur=17 snapshots=23 commits_n=19 additions_n=20 deletions_n=15 files_n=17'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T7"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T7: {failed_count} check(s) failed. See the table above.")
print("T7: ALL CHECKS PASSED")

## T8. Run 2: the same run again

First the Gold rows of run 1 are saved as a copy, so that run 2 can be compared with it. The copy holds the key and the two summary lines of every row.

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {TEST_COPY} AS
    SELECT repo_id, activity_date, event_summary, size_summary
    FROM gold_summary
""")

print("Copy of the Gold rows after run 1:", TEST_COPY)

In [0]:
%run ../01_repository_delivery_metrics $test_schema="test_gold_g1" $source_date="" $run_id="test-run2"

## T9. Checks of run 2: nothing changed

`EXCEPT` returns the rows of the first query that are not in the second one. Both directions must return no rows. The two technical columns are not part of the summary, because they are new in every run.

A run for all dates replaces the whole table, so every row now carries the ID of run 2.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run2'
        UNION ALL
        SELECT '02 Gold rows', CAST(COUNT(*) AS STRING), '8' FROM {TEST_GOLD}
        UNION ALL
        SELECT '03 rows of run 2 that are not in run 1', CAST(COUNT(*) AS STRING), '0'
        FROM (
            SELECT repo_id, activity_date, event_summary, size_summary FROM gold_summary
            EXCEPT
            SELECT repo_id, activity_date, event_summary, size_summary FROM {TEST_COPY}
        )
        UNION ALL
        SELECT '04 rows of run 1 that are not in run 2', CAST(COUNT(*) AS STRING), '0'
        FROM (
            SELECT repo_id, activity_date, event_summary, size_summary FROM {TEST_COPY}
            EXCEPT
            SELECT repo_id, activity_date, event_summary, size_summary FROM gold_summary
        )
        UNION ALL
        SELECT '05 rows written by run 2', CAST(count_if(_gold_run_id = 'test-run2') AS STRING), '8' FROM {TEST_GOLD}
        UNION ALL
        SELECT '06 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '8'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run2'
        UNION ALL
        SELECT '07 audit quality_metrics', MAX(concat('repository_days=', CAST(quality_metrics['repository_days'] AS STRING), ' events=', CAST(quality_metrics['pr_event_count'] AS STRING), ' known=', CAST(quality_metrics['pr_action_known_rows'] AS STRING), ' opened=', CAST(quality_metrics['pr_opened_actions'] AS STRING), ' closed=', CAST(quality_metrics['pr_closed_actions'] AS STRING), ' reopened=', CAST(quality_metrics['pr_reopened_actions'] AS STRING), ' other=', CAST(quality_metrics['pr_other_action_rows'] AS STRING), ' eligible_closed=', CAST(quality_metrics['eligible_closed_rows'] AS STRING), ' merged=', CAST(quality_metrics['merged_closed_rows'] AS STRING), ' merged_dur=', CAST(quality_metrics['merged_duration_eligible_rows'] AS STRING), ' closed_dur=', CAST(quality_metrics['closed_duration_eligible_rows'] AS STRING), ' snapshots=', CAST(quality_metrics['pr_size_snapshot_rows'] AS STRING), ' commits_n=', CAST(quality_metrics['pr_commits_eligible_rows'] AS STRING), ' additions_n=', CAST(quality_metrics['pr_additions_eligible_rows'] AS STRING), ' deletions_n=', CAST(quality_metrics['pr_deletions_eligible_rows'] AS STRING), ' files_n=', CAST(quality_metrics['pr_changed_files_eligible_rows'] AS STRING))),
               'repository_days=8 events=30 known=29 opened=8 closed=19 reopened=1 other=1 eligible_closed=18 merged=12 merged_dur=10 closed_dur=17 snapshots=23 commits_n=19 additions_n=20 deletions_n=15 files_n=17'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run2'
        UNION ALL
        SELECT '08 different runs in the audit table so far', CAST(COUNT(DISTINCT run_id) AS STRING), '3' FROM {TEST_AUDIT}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T9"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T9: {failed_count} check(s) failed. See the table above.")
print("T9: ALL CHECKS PASSED")

## T10. Run 3: late events, for one source date

Three new Silver rows arrive with the source date 2 February:

| Event | Repository | Event date | What it tests |
|---|---|---|---|
| ...32 | 200 | 2 Feb | A second close on a day that already has a Gold row: pull request 22, closed without a merge after 26 hours |
| ...33 | 500 | 2 Feb | A repository that had no Gold row yet |
| ...34 | 200 | **1 Feb** | A late event: it was loaded with the source date 2 February, but it happened on 1 February. The pull request with the number 77 of repository 200 is closed without a merge. This event is now its latest one on that day, so its snapshot changes too |

The run gets `source_date = 2024-02-02`.

In [0]:
spark.sql("INSERT INTO " + TEST_SILVER + """
    (event_id, event_timestamp, event_date, repo_id, repo_name,
     pr_action, pr_id, pr_number, pr_created_at, pr_closed_at, pr_merged_at, pr_merged,
     pr_commits, pr_additions, pr_deletions, pr_changed_files, _source_date)
VALUES
    -- Three new rows with the source date 2 February
    ('36000000032', TIMESTAMP'2024-02-02 11:30:00', DATE'2024-02-02', 200, 'org/beta-renamed', 'closed', 22,   3,  TIMESTAMP'2024-02-01 09:30:00', TIMESTAMP'2024-02-02 11:30:00', NULL, false, 3, 10, 0, 2, DATE'2024-02-02'),
    ('36000000033', TIMESTAMP'2024-02-02 12:00:00', DATE'2024-02-02', 500, 'org/zeta',         'opened', 70,   1,  TIMESTAMP'2024-02-02 12:00:00', NULL,                           NULL, false, 2, 8,  4, 1, DATE'2024-02-02'),
    ('36000000034', TIMESTAMP'2024-02-01 23:59:00', DATE'2024-02-01', 200, 'org/beta',         'closed', NULL, 77, TIMESTAMP'2024-02-01 09:30:00', TIMESTAMP'2024-02-01 23:59:00', NULL, false, 4, 30, 2, 3, DATE'2024-02-02')
""")

print("Silver test rows:", spark.table(TEST_SILVER).count())

In [0]:
%run ../01_repository_delivery_metrics $test_schema="test_gold_g1" $source_date="2024-02-02" $run_id="test-run3"

## T11. Checks of run 3

Worked out by hand.

- The source date 2 February has 6 rows now: ...19, ...20, ...21, ...32, ...33 and ...34. Their event dates are 1 and 2 February, so **two** days are affected. The row without a date belongs to the source date 1 February and is not in the scope of this run.
- Both days are rebuilt from **all** their rows: 18 rows on 1 February (15 + 2 + the late one) and 5 rows on 2 February. 23 rows, 5 repository-days.
- Repository 200 on 1 February has 3 events now: 2 opened and 1 closed without a merge, so the share is 0. Close hours: 09:30 to 23:59 is 14 hours and 29 minutes, 14.4833, rounded 14.48. The pull request with the number 77 has two events on that day. Its snapshot is the late one, with 4 commits, 30 added lines, 2 deleted lines and 3 files. Together with pull request 20: commits (1 + 4) ÷ 2 = 2.5, added lines (0 + 30) ÷ 2 = 15, deleted lines (0 + 2) ÷ 2 = 1, files (1 + 3) ÷ 2 = 2.
- Repository 200 on 2 February has 2 closes, one of them merged: share 50. One merge duration: 24 hours. Close hours 24 and 26: average and median 25. Snapshots of the pull requests 20 and 22: commits (1 + 3) ÷ 2 = 2, added lines (0 + 10) ÷ 2 = 5, deleted lines 0, files (1 + 2) ÷ 2 = 1.5.
- Repository 500 on 2 February is new.
- Repository 100 on 1 February has the same numbers as before, but it was written again by this run.
- The four rows of 5 February are not touched. They still carry the ID of run 2.
- Gold has 5 + 3 = 8 rows.
- Totals of the run: 23 events, 22 with a known action: 8 opened, 12 closed, 1 reopened, 1 other. 11 closes with a known merged flag, 5 of them merged. 3 merge durations and 10 close durations. 16 snapshots: 12 with commits, 13 with added lines, 11 with deleted lines, 10 with files.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '02 audit processing_scope', MAX(processing_scope), 'SINGLE_DATE'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '03 audit source_date', CAST(MAX(source_date) AS STRING), '2024-02-02'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '04 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-01, 2024-02-02]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '05 audit Silver rows read', CAST(MAX(source_rows_read['pull_request_events']) AS STRING), '23'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '06 audit rows without an event date', CAST(MAX(rows_excluded['missing_event_date']) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '07 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '5'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '08 Gold rows', CAST(COUNT(*) AS STRING), '9' FROM {TEST_GOLD}
        UNION ALL
        SELECT '09 rows of 1 and 2 February written by run 3', CAST(count_if(activity_date <= DATE'2024-02-02' AND _gold_run_id = 'test-run3') AS STRING), '5' FROM {TEST_GOLD}
        UNION ALL
        SELECT '10 rows of 5 February still from run 2', CAST(count_if(activity_date = DATE'2024-02-05' AND _gold_run_id = 'test-run2') AS STRING), '4' FROM {TEST_GOLD}
        UNION ALL
        SELECT '11 repository 200 on 1 February (late event): events', MAX(event_summary), 'events=3 known=3 opened=2 closed=1 reopened=0 other=0 eligible_closed=1 merged=0 share=0.0 merged_dur=0 avg_merge=NULL p50_merge=NULL closed_dur=1 avg_close=14.48 p50_close=14.48'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '12 repository 200 on 1 February (late event): sizes', MAX(size_summary), 'snapshots=2 commits_n=2 avg_commits=2.5 additions_n=2 avg_additions=15.0 deletions_n=2 avg_deletions=1.0 files_n=2 avg_files=2.0 name=org/beta'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '13 repository 200 on 2 February (second close): events', MAX(event_summary), 'events=2 known=2 opened=0 closed=2 reopened=0 other=0 eligible_closed=2 merged=1 share=50.0 merged_dur=1 avg_merge=24.0 p50_merge=24.0 closed_dur=2 avg_close=25.0 p50_close=25.0'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '14 repository 200 on 2 February (second close): sizes', MAX(size_summary), 'snapshots=2 commits_n=2 avg_commits=2.0 additions_n=2 avg_additions=5.0 deletions_n=2 avg_deletions=0.0 files_n=2 avg_files=1.5 name=org/beta-renamed'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '15 repository 500 on 2 February (new row): events', MAX(event_summary), 'events=1 known=1 opened=1 closed=0 reopened=0 other=0 eligible_closed=0 merged=0 share=NULL merged_dur=0 avg_merge=NULL p50_merge=NULL closed_dur=0 avg_close=NULL p50_close=NULL'
        FROM gold_summary WHERE repo_id = 500 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '16 repository 500 on 2 February (new row): sizes', MAX(size_summary), 'snapshots=1 commits_n=1 avg_commits=2.0 additions_n=1 avg_additions=8.0 deletions_n=1 avg_deletions=4.0 files_n=1 avg_files=1.0 name=org/zeta'
        FROM gold_summary WHERE repo_id = 500 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '17 repository 100 on 1 February (same numbers): events', MAX(event_summary), 'events=15 known=14 opened=4 closed=8 reopened=1 other=1 eligible_closed=7 merged=4 share=57.14 merged_dur=2 avg_merge=1.17 p50_merge=1.17 closed_dur=6 avg_close=2.56 p50_close=2.0'
        FROM gold_summary WHERE repo_id = 100 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '18 repository 100 on 1 February (same numbers): sizes', MAX(size_summary), 'snapshots=9 commits_n=7 avg_commits=2.29 additions_n=8 avg_additions=28.75 deletions_n=6 avg_deletions=12.17 files_n=5 avg_files=3.6 name=org/alpha-new'
        FROM gold_summary WHERE repo_id = 100 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '19 repository 400 on 2 February (same numbers): events', MAX(event_summary), 'events=2 known=2 opened=1 closed=1 reopened=0 other=0 eligible_closed=1 merged=0 share=0.0 merged_dur=0 avg_merge=NULL p50_merge=NULL closed_dur=1 avg_close=2.5 p50_close=2.5'
        FROM gold_summary WHERE repo_id = 400 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '20 repository 400 on 2 February (same numbers): sizes', MAX(size_summary), 'snapshots=2 commits_n=0 avg_commits=NULL additions_n=0 avg_additions=NULL deletions_n=0 avg_deletions=NULL files_n=0 avg_files=NULL name=NULL'
        FROM gold_summary WHERE repo_id = 400 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '21 repository 300 on 5 February (not touched): events', MAX(event_summary), 'events=5 known=5 opened=0 closed=5 reopened=0 other=0 eligible_closed=5 merged=4 share=80.0 merged_dur=4 avg_merge=4.0 p50_merge=2.5 closed_dur=5 avg_close=7.2 p50_close=3.0'
        FROM gold_summary WHERE repo_id = 300 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '22 repository 300 on 5 February (not touched): sizes', MAX(size_summary), 'snapshots=5 commits_n=5 avg_commits=1.2 additions_n=5 avg_additions=30.0 deletions_n=5 avg_deletions=3.0 files_n=5 avg_files=1.4 name=org/gamma'
        FROM gold_summary WHERE repo_id = 300 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '23 audit quality_metrics', MAX(concat('repository_days=', CAST(quality_metrics['repository_days'] AS STRING), ' events=', CAST(quality_metrics['pr_event_count'] AS STRING), ' known=', CAST(quality_metrics['pr_action_known_rows'] AS STRING), ' opened=', CAST(quality_metrics['pr_opened_actions'] AS STRING), ' closed=', CAST(quality_metrics['pr_closed_actions'] AS STRING), ' reopened=', CAST(quality_metrics['pr_reopened_actions'] AS STRING), ' other=', CAST(quality_metrics['pr_other_action_rows'] AS STRING), ' eligible_closed=', CAST(quality_metrics['eligible_closed_rows'] AS STRING), ' merged=', CAST(quality_metrics['merged_closed_rows'] AS STRING), ' merged_dur=', CAST(quality_metrics['merged_duration_eligible_rows'] AS STRING), ' closed_dur=', CAST(quality_metrics['closed_duration_eligible_rows'] AS STRING), ' snapshots=', CAST(quality_metrics['pr_size_snapshot_rows'] AS STRING), ' commits_n=', CAST(quality_metrics['pr_commits_eligible_rows'] AS STRING), ' additions_n=', CAST(quality_metrics['pr_additions_eligible_rows'] AS STRING), ' deletions_n=', CAST(quality_metrics['pr_deletions_eligible_rows'] AS STRING), ' files_n=', CAST(quality_metrics['pr_changed_files_eligible_rows'] AS STRING))),
               'repository_days=5 events=23 known=22 opened=8 closed=12 reopened=1 other=1 eligible_closed=11 merged=5 merged_dur=3 closed_dur=10 snapshots=16 commits_n=12 additions_n=13 deletions_n=11 files_n=10'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '24 different keys in Gold', CAST(COUNT(DISTINCT repo_id, activity_date) AS STRING), '9' FROM {TEST_GOLD}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T11"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T11: {failed_count} check(s) failed. See the table above.")
print("T11: ALL CHECKS PASSED")

## T12. Run 4: removed events

Two Silver rows are removed: the second close of repository 200 on 2 February (...32) and the only event of repository 500 (...33). Then the source date 2 February runs again.

In [0]:
spark.sql(f"""
    DELETE FROM {TEST_SILVER}
    WHERE event_id IN ('36000000032', '36000000033')
""")

print("Silver test rows:", spark.table(TEST_SILVER).count())

In [0]:
%run ../01_repository_delivery_metrics $test_schema="test_gold_g1" $source_date="2024-02-02" $run_id="test-run4"

## T13. Checks of run 4

Worked out by hand.

- The source date 2 February has 4 rows left: ...19, ...20, ...21 and the late event ...34. The affected days are still 1 and 2 February.
- 18 rows on 1 February and 3 rows on 2 February: 21 rows, 4 repository-days.
- Repository 200 on 2 February is back to one merge after 24 hours.
- Repository 500 has no event any more, so its Gold row is gone.
- Gold has 4 + 4 = 8 rows. The rows of 5 February still carry the ID of run 2.
- Totals of the run: 21 events, 20 with a known action: 7 opened, 11 closed, 1 reopened, 1 other. 10 closes with a known merged flag, 5 of them merged. 3 merge durations and 9 close durations. 14 snapshots: 10 with commits, 11 with added lines, 9 with deleted lines, 8 with files.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '02 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-01, 2024-02-02]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '03 audit Silver rows read', CAST(MAX(source_rows_read['pull_request_events']) AS STRING), '21'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '04 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '4'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '05 Gold rows', CAST(COUNT(*) AS STRING), '8' FROM {TEST_GOLD}
        UNION ALL
        SELECT '06 rows of repository 500 (removed)', CAST(count_if(repo_id = 500) AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '07 repository 200 on 2 February (lowered): events', MAX(event_summary), 'events=1 known=1 opened=0 closed=1 reopened=0 other=0 eligible_closed=1 merged=1 share=100.0 merged_dur=1 avg_merge=24.0 p50_merge=24.0 closed_dur=1 avg_close=24.0 p50_close=24.0'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '08 repository 200 on 2 February (lowered): sizes', MAX(size_summary), 'snapshots=1 commits_n=1 avg_commits=1.0 additions_n=1 avg_additions=0.0 deletions_n=1 avg_deletions=0.0 files_n=1 avg_files=1.0 name=org/beta-renamed'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-02'
        UNION ALL
        SELECT '09 repository 200 on 1 February (late event still there): events', MAX(event_summary), 'events=3 known=3 opened=2 closed=1 reopened=0 other=0 eligible_closed=1 merged=0 share=0.0 merged_dur=0 avg_merge=NULL p50_merge=NULL closed_dur=1 avg_close=14.48 p50_close=14.48'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '10 repository 200 on 1 February (late event still there): sizes', MAX(size_summary), 'snapshots=2 commits_n=2 avg_commits=2.5 additions_n=2 avg_additions=15.0 deletions_n=2 avg_deletions=1.0 files_n=2 avg_files=2.0 name=org/beta'
        FROM gold_summary WHERE repo_id = 200 AND activity_date = DATE'2024-02-01'
        UNION ALL
        SELECT '11 rows of 1 and 2 February written by run 4', CAST(count_if(activity_date <= DATE'2024-02-02' AND _gold_run_id = 'test-run4') AS STRING), '4' FROM {TEST_GOLD}
        UNION ALL
        SELECT '12 rows of 5 February still from run 2', CAST(count_if(activity_date = DATE'2024-02-05' AND _gold_run_id = 'test-run2') AS STRING), '4' FROM {TEST_GOLD}
        UNION ALL
        SELECT '13 audit quality_metrics', MAX(concat('repository_days=', CAST(quality_metrics['repository_days'] AS STRING), ' events=', CAST(quality_metrics['pr_event_count'] AS STRING), ' known=', CAST(quality_metrics['pr_action_known_rows'] AS STRING), ' opened=', CAST(quality_metrics['pr_opened_actions'] AS STRING), ' closed=', CAST(quality_metrics['pr_closed_actions'] AS STRING), ' reopened=', CAST(quality_metrics['pr_reopened_actions'] AS STRING), ' other=', CAST(quality_metrics['pr_other_action_rows'] AS STRING), ' eligible_closed=', CAST(quality_metrics['eligible_closed_rows'] AS STRING), ' merged=', CAST(quality_metrics['merged_closed_rows'] AS STRING), ' merged_dur=', CAST(quality_metrics['merged_duration_eligible_rows'] AS STRING), ' closed_dur=', CAST(quality_metrics['closed_duration_eligible_rows'] AS STRING), ' snapshots=', CAST(quality_metrics['pr_size_snapshot_rows'] AS STRING), ' commits_n=', CAST(quality_metrics['pr_commits_eligible_rows'] AS STRING), ' additions_n=', CAST(quality_metrics['pr_additions_eligible_rows'] AS STRING), ' deletions_n=', CAST(quality_metrics['pr_deletions_eligible_rows'] AS STRING), ' files_n=', CAST(quality_metrics['pr_changed_files_eligible_rows'] AS STRING))),
               'repository_days=4 events=21 known=20 opened=7 closed=11 reopened=1 other=1 eligible_closed=10 merged=5 merged_dur=3 closed_dur=9 snapshots=14 commits_n=10 additions_n=11 deletions_n=9 files_n=8'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run4'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T13"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T13: {failed_count} check(s) failed. See the table above.")
print("T13: ALL CHECKS PASSED")

## T14. Run 5: a source date without rows

No Silver row has the source date 3 February. The run gets `source_date = 2024-02-03`.

In [0]:
%run ../01_repository_delivery_metrics $test_schema="test_gold_g1" $source_date="2024-02-03" $run_id="test-run5"

## T15. Checks of run 5: `NO_INPUT`, Gold unchanged

There is no affected day, so nothing is published. The Gold table still holds the 4 rows of run 4 and the 4 rows of run 2. The run still writes its audit row.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'NO_INPUT' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '02 audit processing_scope', MAX(processing_scope), 'SINGLE_DATE'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '03 audit source_date', CAST(MAX(source_date) AS STRING), '2024-02-03'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '04 audit number of affected dates', CAST(MAX(size(affected_dates)) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '05 audit Silver rows read', CAST(MAX(source_rows_read['pull_request_events']) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '06 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '07 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '08 audit error_message', MAX(error_message), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '09 audit repository_days', CAST(MAX(quality_metrics['repository_days']) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '10 Gold rows (unchanged)', CAST(COUNT(*) AS STRING), '8' FROM {TEST_GOLD}
        UNION ALL
        SELECT '11 Gold rows written by run 5', CAST(count_if(_gold_run_id = 'test-run5') AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '12 Gold rows still from run 4', CAST(count_if(_gold_run_id = 'test-run4') AS STRING), '4' FROM {TEST_GOLD}
        UNION ALL
        SELECT '13 Gold rows still from run 2', CAST(count_if(_gold_run_id = 'test-run2') AS STRING), '4' FROM {TEST_GOLD}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T15"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T15: {failed_count} check(s) failed. See the table above.")
print("T15: ALL CHECKS PASSED")

## T16. Run 6: a quality gate fails

The Silver notebook never lets a row without `repo_id` into Silver. The test puts one such row into the test table anyway, on 5 February. Its group has no repository, so the Gold key would be NULL. The gate `KEY_IS_NULL` must catch it.

The cell below prepares the fault. The run itself is in the cell after it.

In [0]:
spark.sql("INSERT INTO " + TEST_SILVER + """
    (event_id, event_timestamp, event_date, repo_id, repo_name,
     pr_action, pr_id, pr_number, pr_created_at, pr_closed_at, pr_merged_at, pr_merged,
     pr_commits, pr_additions, pr_deletions, pr_changed_files, _source_date)
VALUES
    -- A row without repo_id
    ('36000000035', TIMESTAMP'2024-02-05 13:00:00', DATE'2024-02-05', NULL, 'org/broken', 'opened', 80, 1, TIMESTAMP'2024-02-05 13:00:00', NULL, NULL, false, 1, 1, 1, 1, DATE'2024-02-05')
""")

print("Silver test rows:", spark.table(TEST_SILVER).count())

### Run 6

**The next cell is EXPECTED to fail** with `RuntimeError: repository_delivery_metrics FAILED: Quality gates failed: KEY_IS_NULL`. After it has failed, click the cell after it and choose "Run all below".

If this cell runs twice, the audit table gets two rows for run 6. The checks count runs, not rows, so that does no harm.

In [0]:
%run ../01_repository_delivery_metrics $test_schema="test_gold_g1" $source_date="2024-02-05" $run_id="test-run6-fail"

## T17. Checks of run 6: Gold is unchanged

Worked out by hand.

- The source date 5 February has 11 rows: the 10 good ones and the bad one. The affected day is 5 February.
- The run staged 5 rows: repositories 300, 310, 320 and 600, and one row without a repository.
- It must have written its audit row with the status `FAILED` and the name of the gate.
- The Gold table must be exactly what it was: 8 rows, and the rows of 5 February still carry the ID of run 2. The rejected result is only in the staging table.

After the checks, the test removes the bad row, so that the next run can succeed.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'FAILED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '02 audit failed_checks', MAX(failed_checks), 'KEY_IS_NULL'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '03 audit error_message', MAX(error_message), 'Quality gates failed: KEY_IS_NULL'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '04 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'false'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '05 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '06 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-05]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '07 audit Silver rows read', CAST(MAX(source_rows_read['pull_request_events']) AS STRING), '11'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '08 Gold rows (unchanged)', CAST(COUNT(*) AS STRING), '8' FROM {TEST_GOLD}
        UNION ALL
        SELECT '09 Gold rows written by the failed run', CAST(count_if(_gold_run_id = 'test-run6-fail') AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '10 Gold rows of 5 February still from run 2', CAST(count_if(activity_date = DATE'2024-02-05' AND _gold_run_id = 'test-run2') AS STRING), '4' FROM {TEST_GOLD}
        UNION ALL
        SELECT '11 Gold rows without repo_id', CAST(count_if(repo_id IS NULL) AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '12 staging rows of the rejected result', CAST(count_if(_gold_run_id = 'test-run6-fail') AS STRING), '5' FROM {TEST_STAGING}
        UNION ALL
        SELECT '13 staging rows without repo_id', CAST(count_if(repo_id IS NULL) AS STRING), '1' FROM {TEST_STAGING}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T17"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T17: {failed_count} check(s) failed. See the table above.")
print("T17: ALL CHECKS PASSED")

In [0]:
spark.sql(f"""
    DELETE FROM {TEST_SILVER}
    WHERE event_id = '36000000035'
""")

print("Silver test rows after the repair:", spark.table(TEST_SILVER).count())

## T18. Run 7: recovery

The same run again, now without the bad row. The four rows of 5 February are replaced. The four rows of 1 and 2 February are not touched and still carry the ID of run 4.

In [0]:
%run ../01_repository_delivery_metrics $test_schema="test_gold_g1" $source_date="2024-02-05" $run_id="test-run7"

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '02 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '4'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '03 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '04 audit failed_checks', MAX(failed_checks), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '05 Gold rows', CAST(COUNT(*) AS STRING), '8' FROM {TEST_GOLD}
        UNION ALL
        SELECT '06 rows of 5 February written by run 7', CAST(count_if(activity_date = DATE'2024-02-05' AND _gold_run_id = 'test-run7') AS STRING), '4' FROM {TEST_GOLD}
        UNION ALL
        SELECT '07 rows of 1 and 2 February still from run 4', CAST(count_if(activity_date <= DATE'2024-02-02' AND _gold_run_id = 'test-run4') AS STRING), '4' FROM {TEST_GOLD}
        UNION ALL
        SELECT '08 repository 310 on 5 February: events', MAX(event_summary), 'events=1 known=1 opened=0 closed=1 reopened=0 other=0 eligible_closed=1 merged=1 share=100.0 merged_dur=1 avg_merge=1584.0 p50_merge=1584.0 closed_dur=1 avg_close=1584.25 p50_close=1584.25'
        FROM gold_summary WHERE repo_id = 310 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '09 repository 310 on 5 February: sizes', MAX(size_summary), 'snapshots=1 commits_n=1 avg_commits=12.0 additions_n=1 avg_additions=3456.0 deletions_n=1 avg_deletions=789.0 files_n=1 avg_files=25.0 name=org/delta'
        FROM gold_summary WHERE repo_id = 310 AND activity_date = DATE'2024-02-05'
        UNION ALL
        SELECT '10 different runs in the audit table so far', CAST(COUNT(DISTINCT run_id) AS STRING), '8' FROM {TEST_AUDIT}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T18"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T18: {failed_count} check(s) failed. See the table above.")
print("T18: ALL CHECKS PASSED")

## T19. Run 8: a day that no longer exists in Silver

All Silver rows of 5 February are removed. A run for the source date 5 February would now find no rows and end as `NO_INPUT`, as in T15, and the three Gold rows of that day would stay. The way to remove such a day is a run for all dates: it replaces the whole table.

In [0]:
spark.sql(f"""
    DELETE FROM {TEST_SILVER}
    WHERE event_date = DATE'2024-02-05'
""")

print("Silver test rows:", spark.table(TEST_SILVER).count())

In [0]:
%run ../01_repository_delivery_metrics $test_schema="test_gold_g1" $source_date="" $run_id="test-run8"

## T20. Checks of run 8

Worked out by hand: Silver has 22 rows left. 21 have an event date, on 1 and 2 February, and 1 has none. Gold has the 4 rows of those two days and no row of 5 February.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '02 audit processing_scope', MAX(processing_scope), 'ALL_DATES'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '03 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-01, 2024-02-02]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '04 audit Silver rows read', CAST(MAX(source_rows_read['pull_request_events']) AS STRING), '21'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '05 audit rows without an event date', CAST(MAX(rows_excluded['missing_event_date']) AS STRING), '1'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '06 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '4'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '07 Gold rows', CAST(COUNT(*) AS STRING), '4' FROM {TEST_GOLD}
        UNION ALL
        SELECT '08 Gold rows of 5 February', CAST(count_if(activity_date = DATE'2024-02-05') AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '09 rows written by run 8', CAST(count_if(_gold_run_id = 'test-run8') AS STRING), '4' FROM {TEST_GOLD}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T20"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T20: {failed_count} check(s) failed. See the table above.")
print("T20: ALL CHECKS PASSED")

## T21. Bad parameter: `source_date`

`2024-2-3` is not in the form `YYYY-MM-DD`. The notebook must stop in its parameter step, before it reads or writes a table.

**The next cell is EXPECTED to fail** with `ValueError: Invalid source_date '2024-2-3'`. After it has failed, click the cell after it and choose "Run all below".

In [0]:
%run ../01_repository_delivery_metrics $test_schema="test_gold_g1" $source_date="2024-2-3" $run_id="test-bad-date"

### T21 check: where did it stop?

`%run` runs the other notebook inside this one, so the variables of that notebook can be read here. They show how far the stopped run came:

- `source_date` holds the bad value, so step 1.2 has run.
- `run_id` is still the ID of run 8, so the notebook never came to step 1.4, where it reads the run ID. It stopped before it touched a table.

In [0]:
print("source_date:", source_date)
print("run_id     :", run_id)

if source_date != "2024-2-3":
    raise AssertionError("T21: the run with the bad source_date has not run. Run the cell above first.")

if run_id == "test-bad-date":
    raise AssertionError("T21: the notebook did not stop at the bad source_date. It went on to step 1.4.")

section_results["T21"] = 0
print("T21: the notebook stopped at the bad source_date, as expected")

## T22. Bad parameter: `test_schema`

`TEST_WRONG` has capital letters, so it is not a valid test schema name.

**The next cell is EXPECTED to fail** with `ValueError: Invalid test_schema 'TEST_WRONG'`. After it has failed, click the cell after it and choose "Run all below".

In [0]:
%run ../01_repository_delivery_metrics $test_schema="TEST_WRONG" $source_date="" $run_id="test-bad-schema"

### T22 check: where did it stop?

- `test_schema` holds the bad value, so step 1.3 has run.
- `run_id` is still the ID of run 8, so the notebook never came to step 1.4.

In [0]:
print("test_schema:", test_schema)
print("run_id     :", run_id)

if test_schema != "TEST_WRONG":
    raise AssertionError("T22: the run with the bad test_schema has not run. Run the cell above first.")

if run_id == "test-bad-schema":
    raise AssertionError("T22: the notebook did not stop at the bad test_schema. It went on to step 1.4.")

section_results["T22"] = 0
print("T22: the notebook stopped at the bad test_schema, as expected")

## T23. Bad parameter: `run_id`

The run ID is written into every Gold row, so it may contain only letters, digits, `-` and `_`. `bad.run.id` contains dots.

**The next cell is EXPECTED to fail** with `ValueError: Invalid run_id 'bad.run.id'`. After it has failed, click the cell after it and choose "Run all below".

In [0]:
%run ../01_repository_delivery_metrics $test_schema="test_gold_g1" $source_date="" $run_id="bad.run.id"

### T23 check: did it run?

- `run_id` holds the bad value, so the notebook came to step 1.4.
- That it stopped there, and wrote nothing, is checked in T24: this run has no audit row and the Gold table is still the one of run 8.

In [0]:
print("test_schema:", test_schema)
print("run_id     :", run_id)

if run_id != "bad.run.id":
    raise AssertionError("T23: the run with the bad run_id has not run. Run the cell above first.")

section_results["T23"] = 0
print("T23: the notebook came to the run_id step. T24 checks that it wrote nothing.")

## T24. Final checks: nothing was written by the three stopped runs, and production is untouched

- The three runs with a bad parameter stopped before the audit step, so the audit table still holds 9 runs and the Gold table is still the one of run 8.
- Every run wrote exactly one audit row. Run 6 is left out of that check, because its cell may have run more than once.
- The versions of the production tables are read again and compared with the versions from T2.

In [0]:
prod_audit_version_after = spark.sql(
    f"DESCRIBE HISTORY {PROD_AUDIT} LIMIT 1"
).collect()[0]["version"]

if spark.catalog.tableExists(PROD_GOLD):
    prod_gold_version_after = spark.sql(
        f"DESCRIBE HISTORY {PROD_GOLD} LIMIT 1"
    ).collect()[0]["version"]
else:
    prod_gold_version_after = -1

if spark.catalog.tableExists(PROD_STAGING):
    prod_staging_version_after = spark.sql(
        f"DESCRIBE HISTORY {PROD_STAGING} LIMIT 1"
    ).collect()[0]["version"]
else:
    prod_staging_version_after = -1

print("Production audit version  :", prod_audit_version_after)
print("Production Gold version   :", prod_gold_version_after)
print("Production staging version:", prod_staging_version_after)

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 different runs in the audit table' AS check_name, CAST(COUNT(DISTINCT run_id) AS STRING) AS actual, '9' AS expected
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '02 audit rows of the three stopped runs', CAST(count_if(run_id IN ('test-bad-date', 'test-bad-schema', 'bad.run.id')) AS STRING), '0'
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '03 Gold rows still from run 8', CAST(count_if(_gold_run_id = 'test-run8') AS STRING), '4'
        FROM {TEST_GOLD}
        UNION ALL
        SELECT '04 Gold rows', CAST(COUNT(*) AS STRING), '4'
        FROM {TEST_GOLD}
        UNION ALL
        SELECT '05 runs by status',
               concat('SUCCEEDED=', CAST(COUNT(DISTINCT CASE WHEN status = 'SUCCEEDED' THEN run_id END) AS STRING),
                      ' FAILED=', CAST(COUNT(DISTINCT CASE WHEN status = 'FAILED' THEN run_id END) AS STRING),
                      ' NO_INPUT=', CAST(COUNT(DISTINCT CASE WHEN status = 'NO_INPUT' THEN run_id END) AS STRING)),
               'SUCCEEDED=6 FAILED=1 NO_INPUT=2'
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '06 production audit table version unchanged', '{prod_audit_version_after}', '{prod_audit_version_before}'
        UNION ALL
        SELECT '07 production Gold table version unchanged', '{prod_gold_version_after}', '{prod_gold_version_before}'
        UNION ALL
        SELECT '08 production staging version unchanged', '{prod_staging_version_after}', '{prod_staging_version_before}'
        UNION ALL
        SELECT '09 audit rows of the 8 runs without run 6 (one row each)', CAST(count_if(run_id <> 'test-run6-fail') AS STRING), '8'
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '10 audit rows of run 6 are all FAILED', CAST(count_if(run_id = 'test-run6-fail' AND status <> 'FAILED') AS STRING), '0'
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '11 audit rows with another pipeline or table name', CAST(count_if(pipeline_name <> 'gold_repository_delivery_metrics' OR gold_table <> '{TEST_GOLD}') AS STRING), '0'
        FROM {TEST_AUDIT}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T24"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T24: {failed_count} check(s) failed. See the table above.")
print("T24: ALL CHECKS PASSED")

## Result of the whole test

Every section wrote its number of failed checks into `section_results`. This cell goes through the 13 sections. A section that is not in the dictionary did not run, for example because the cells above it were skipped.

The test has passed only when this cell prints `TEST PASSED`.

In [0]:
EXPECTED_SECTIONS = [
    "T4", "T7", "T9", "T11", "T13", "T15", "T17",
    "T18", "T20", "T21", "T22", "T23", "T24",
]

problems = []

for section in EXPECTED_SECTIONS:
    if section not in section_results:
        problems.append(section + " did not run")
    elif section_results[section] > 0:
        problems.append(section + " has " + str(section_results[section]) + " failed check(s)")

print("Sections that ran:", len(section_results), "of", len(EXPECTED_SECTIONS))

if len(problems) > 0:
    raise AssertionError("TEST FAILED: " + "; ".join(problems))

print("TEST PASSED: all", len(EXPECTED_SECTIONS), "sections ran and every check passed.")

## Clean up: drop the test schema

Only a schema whose name starts with `test_` is ever dropped.

If the test failed, you can leave this cell out and look at the test tables first. The next test run drops the schema anyway, in T1.

In [0]:
if not TEST_SCHEMA.startswith("test_"):
    raise RuntimeError("Refusing to drop a schema that is not a test schema.")

spark.sql(f"DROP SCHEMA IF EXISTS github_lakehouse.{TEST_SCHEMA} CASCADE")

print("Dropped:", TEST_SCHEMA)